In [1]:
import os
import pickle
import dill
import pprint
import itertools
import pathos
import pprint
import functools
from functools import partial
from pathlib import Path
import shutil

import pandas as pd
import numpy as np
from scipy import stats as st

import plotly.graph_objects as go
from matplotlib import pyplot as plt
import seaborn as sns
sns.set()

import sys
sys.path.append('/Users/leonardo.tessarolo/git/bayesian_bss/')

from src import MMSEMetropolisHastingsEstimator, MAPGradientAscentEstimator, InstantaneousMixtureModel, PosteriorContourLines, ContourLineGraphPlotter, MCMCGraphPlotter, MAPGradientAscentGraphPlotter, ExponentialPrior, LogisticSource, BayesianEstimators, TriangularSource, ExperimentExecutor

print(os.cpu_count())

np.random.seed(2000)

8


In [2]:
# Whether or not to create config
CREATE_CONFIG=True

# Whether or not to create folder structure
CREATE_FOLDER_STRUCTURE=True

# Experiment name
EXPERIMENT_NAME='test_complete_refit_contours'
NEW_EXPERIMENT_NAME='test_complete_refit_contours_v2'

# Folder which will contain output directory tree
OUTPUT_DIR='./output'
base_output_path=Path(OUTPUT_DIR)

# Create experiment directory
experiment_dir = base_output_path / EXPERIMENT_NAME
new_experiment_dir = base_output_path / NEW_EXPERIMENT_NAME

# 1. Configurations

In [3]:
# Number of sources and observations
NSOURCES=2
NOBS=1000

# Mixing matrix configuration
A = np.array([
    [1, 1],
    [-0.5, 0.5]
])
# A = np.array([
#     [1/0.98, 0],
#     [0, 0.98]
# ])

# Define initial conditions for optimizations
initial_B = np.linalg.inv(
    A
) + np.random.normal(
    0,0.1,
    A.shape
)
initial_B = initial_B.reshape(
    (NSOURCES, NSOURCES, 1)
)

# Execution configurations
exec_configs = {
    'general': {
        # Experiment name for folder structure
        'experiment_name': NEW_EXPERIMENT_NAME,
        # Experiment directory
        'experiment_dir': new_experiment_dir,
        # Whether or not to create folder structure
        'create_folder_structure': CREATE_FOLDER_STRUCTURE,
        # Whether or not to save graphs
        'save_graphs': False,
        # Number of sources
        'n_sources': NSOURCES,
        # Number of observations in each realization
        'n_obs': NOBS,
        # Number of parallel workers
        'n_workers': os.cpu_count(),
        # Number of parallel realizations to run
        'n_realizations': 100,
        # Mixing matrix
        'A': A,
        # Separating matrix
        'B': np.linalg.inv(A),
        # Initial condition
        'initial_B': initial_B,
        # Whether or not to use a normalized posterior
        'normalize_posterior': True
    },
    'contour': {
        # Number of points used in contour line grids
        'contour_grid_points': 401,
        # Exploration limits for grids
        'u_lims':(-0.4, 0.4),
        'v_lims':(-0.4, 0.4),
        'central_point':(0.0, 0.0)
    },
    'map': {
        # Threshold for stopping optimization
        'stopping_thresh': 1E-7,
        # Maximum number of iterations
        'max_it': 1000000,
        # Learning rate
        'learning_rate': 1E-4,
        # Persistance iterations for stopping criterion
        'stopping_criterion_persistance_its': 20
    },
    'mcmc': {
        # Exploration variance for MCMC
        'exploration_var': 5E-5,
        # Number of samples to generate
        'n_samples': 30000,
        # Burn-in samples
        'burn_in': 0.5
    },
    'sim': {
        'source_model': LogisticSource(
            mu=0.0,
            sigma=1.0
        ),
        'sources': {
            'perfect_model': LogisticSource(
                mu=0.0,
                sigma=1.0
            ),
            'slightly_misspecified_model': LogisticSource(
                mu=0.1,
                sigma=1.1
            ),
            'largely_misspecified_model': TriangularSource(
                lower=-2,
                upper=2,
                mode=0
            )
        },
        'priors': {
            'likelihood': ExponentialPrior(
                center=np.linalg.inv(A),
                std=np.inf
            ),
            'non_informative_prior': ExponentialPrior(
                center=np.linalg.inv(A),
                std=10
            ),
            'informative_prior': ExponentialPrior(
                center=np.linalg.inv(A),
                std=0.1
            ),
            'identity_transform': ExponentialPrior(
                center=np.eye(NSOURCES),
                std=0.1
            )
        },
        'test_cases': {
            'i': {
                'source': 'perfect_model',
                'prior': 'likelihood',
            },
            'ii': {
                'source': 'perfect_model',
                'prior': 'non_informative_prior',
            },
            'iii': {
                'source': 'perfect_model',
                'prior': 'informative_prior',
            },
            'iv': {
                'source': 'perfect_model',
                'prior': 'identity_transform',
            },
            'v': {
                'source': 'slightly_misspecified_model',
                'prior': 'likelihood',
            },
            'vi': {
                'source': 'slightly_misspecified_model',
                'prior': 'non_informative_prior',
            },
            'vii': {
                'source': 'slightly_misspecified_model',
                'prior': 'informative_prior',
            },
            'viii': {
                'source': 'slightly_misspecified_model',
                'prior': 'identity_transform',
            },
            'ix': {
                'source': 'largely_misspecified_model',
                'prior': 'likelihood',
            },
            'x': {
                'source': 'largely_misspecified_model',
                'prior': 'non_informative_prior',
            },
            'xi': {
                'source': 'largely_misspecified_model',
                'prior': 'informative_prior',
            },
            'xii': {
                'source': 'largely_misspecified_model',
                'prior': 'identity_transform',
            },
        }
    }
}

pprint.pp(exec_configs)

{'general': {'experiment_name': 'test_complete_refit_contours_v2',
             'experiment_dir': PosixPath('output/test_complete_refit_contours_v2'),
             'create_folder_structure': True,
             'save_graphs': False,
             'n_sources': 2,
             'n_obs': 1000,
             'n_workers': 8,
             'n_realizations': 100,
             'A': array([[ 1. ,  1. ],
       [-0.5,  0.5]]),
             'B': array([[ 0.5, -1. ],
       [ 0.5,  1. ]]),
             'initial_B': array([[[ 0.67367376],
        [-0.81020861]],

       [[ 0.28932266],
        [ 0.98510879]]]),
             'normalize_posterior': True},
 'contour': {'contour_grid_points': 401,
             'u_lims': (-0.4, 0.4),
             'v_lims': (-0.4, 0.4),
             'central_point': (0.0, 0.0)},
 'map': {'stopping_thresh': 1e-07,
         'max_it': 1000000,
         'learning_rate': 0.0001,
         'stopping_criterion_persistance_its': 20},
 'mcmc': {'exploration_var': 5e-05, 'n_samples': 30

# 2. Folder Structure

In [4]:
# Initialize folder structure, if so specified
if CREATE_FOLDER_STRUCTURE:
    if exec_configs['general']['create_folder_structure']:
        # Creates base output path and experiment dir
        if not base_output_path.is_dir():
            base_output_path.mkdir()
        if not new_experiment_dir.is_dir():
            new_experiment_dir.mkdir()
        
        shutil.copytree(
             experiment_dir,
             new_experiment_dir,
             dirs_exist_ok=True
        )

        # Iter realizations and create success flags
        for r in range(exec_configs['general']['n_realizations']):
            # Overall folder for realization
            realization_dir = new_experiment_dir / str(r)
            
            with (realization_dir/'success_flag.pkl').open('wb') as f:
                    dill.dump('UNFINISHED', f)
        

if CREATE_CONFIG:
     with (new_experiment_dir/'execution_config.pkl').open('wb') as f:
        dill.dump(exec_configs, f)


# 3. Execute Experiment

In [5]:
ex = ExperimentExecutor(
    cfg=exec_configs,
    initialize=False
)

In [ ]:
%%time
ex.rerun_contour(
    test_cases = ['i','ii','iii','iv','v','vi','vii','viii']
)

####################################################################################################
Total realizations: 100
Finished realizations: 0
Unfinished realizations: 100
####################################################################################################


In [ ]:
PARA

NameError: name 'PARA' is not defined

In [ ]:
realization_dir=new_experiment_dir/'0'

with (realization_dir/'results_raw.pkl').open('rb') as f:
    t=dill.load(f)

In [ ]:
p = t['results']['i']['mmse_estimator']
p.__dir__()

['n_samples',
 'log_posterior_fn',
 'Q',
 'burn_in',
 'burn_in_start',
 'B_est_idx',
 'B_est',
 'mcmc_results',
 '__module__',
 '__firstlineno__',
 '__init__',
 '_MMSEMetropolisHastingsEstimator__get_Q',
 '_MMSEMetropolisHastingsEstimator__get_log_posterior_fn',
 'fit',
 '__static_attributes__',
 '__dict__',
 '__weakref__',
 '__doc__',
 '__slotnames__',
 '__new__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

(30000, 2, 2)